# Pipeline — multi-year intra-US MRIOT build & save  ⚠️ LEGACY (v2 schema)

> **⚠️ Do NOT use this notebook to (re)generate production tables.**
> It builds the **superseded v2 aggregation variant** (`reconstruct_bilateral_2`
> with the `nd0+nm0` import marginal, `include_dm0`, `basic_prices`, `balance`
> switch) — the method that `comparison.ipynb` showed is dominated by the v3
> *md0-chain* build on every identity metric ("v3 ≫ v2").
> The **production multi-year build lives in [`v3_construction.ipynb`](v3_construction.ipynb)**:
> set `YEARS` to the full list (now the default) and run all — it builds
> `grav_fric_v3.1`, applies the row-preserving column RAS (`ras_one_year`, all
> years) and the OECD/SNA VA convention (all years), producing a complete,
> schema-homogeneous `grav_fric_v3.1_RAS`.
> This notebook is kept only to reproduce the historical `grav_fric_v1.2` build
> for comparisons.

Builds the WiNDC-derived MRIOT for the selected years with `build_mriot_for_year` and (optionally) saves each year to disk. Construction follows the **v2 aggregation** schema (`reconstruct_bilateral_2` with the `nd0+nm0` import marginal, `include_dm0` folding local margins in, `basic_prices`, and a `balance` switch).

Only what the pipeline actually uses is kept here; the construction/comparison exploration lives in `v2_construction`, `v3_construction` and `comparison`.

## Setup

# Librairies

#### Installations

In [ ]:
from paths import ROOT
import sys
!{sys.executable} -m pip install gdx2py

In [ ]:
import sys
!{sys.executable} -m pip install gamspy-base

#### Imports

In [ ]:
from gdx2py import GdxFile
import gamspy_base
import os
import pandas as pd
from gdx2py.gams import GAMSParameter, GAMSSet
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from itertools import product

In [ ]:
gamspybase_directory = gamspy_base.directory
print(gamspybase_directory)

In [ ]:
path_windc_gdx = str(ROOT / "data/raw/GTAPWiNDC/data/core/WiNDCdatabase.gdx")

In [ ]:
gdx = GdxFile(path_windc_gdx, gams_dir=gamspybase_directory)
print(list(gdx))

In [ ]:
# ── 1. Load all parameters ───────────────────────────────────────────────────
params = {}
for name, obj in gdx:
    if isinstance(obj, GAMSParameter):
        s = obj.to_pandas()
        if s is not None and len(s) > 0:
            df = s.reset_index()
            df.columns = list(df.columns[:-1]) + ['value']
            params[name] = df
# params contains all years and regions. We will filter it later when we need to build the IOT for a specific year and region.

In [ ]:
# Regions: union of all states present in xn0_ or nd0_
all_xn0 = set(params['xn0_']['r'].unique())
all_nd0 = set(params['nd0_']['r'].unique())
regions = sorted(all_xn0 | all_nd0)
n = len(regions)
print(n)

In [ ]:
# Economic (GDP-weighted) centroids -- the delivered reference points of the gravity
# distance matrix. Built by 02_economic_centroids.ipynb (BEA CAGDP2 county GDP + the
# 2020 Census county population centroids). Loaded here as {abbr: (lat, lon)}.
_cen = pd.read_csv(ROOT / "data/interim/economic_centroids.csv")
COORDS = {r.abbr: (r.lat, r.lon) for r in _cen.itertuples(index=False)}


In [ ]:
# distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlam = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [ ]:
# Distance matrix between all pairs of regions, great-circle (haversine) between the
# GDP-weighted ECONOMIC CENTROIDS of each region (data/interim/economic_centroids.csv,
# built in 02_economic_centroids.ipynb). These are the delivered reference points; the
# earlier capital-based prior and the alternative (geometric, population-weighted)
# centroids are compared in analysis/distance_variants.py of the source project.
D= pd.DataFrame(index=regions, columns=regions, dtype=float)
for i, j in product(regions, regions):
    D.loc[i, j] = np.nan if i == j else haversine(*COORDS[i], *COORDS[j])

D_np = D.values.copy()

missing = [r for r in regions if r not in COORDS]
if missing:
    print(f"Warning: regions without coordinates: {missing}")
print(f"{len(regions)} regions | distance range: "
      f"{D_np[~np.isnan(D_np)].min():.0f}-{D_np[~np.isnan(D_np)].max():.0f} km")


In [ ]:
# Example: load use matrix for New York 2017 as a numpy array (goods × sectors)
def load_matrix(param_name, year, regions, sectors):
    df = params[param_name]
    dim = 'g' if 'g' in df.columns else 's'
    return (df[df['yr'] == year]
            .groupby(['r', dim])['value'].sum()
            .unstack(dim)
            .reindex(index=regions, columns=sectors, fill_value=0.0)
            .fillna(0.0)          
            .values)

In [ ]:
def load_year_data(year, regions, sectors, names_params):
    """Load all IO matrices for a given year. Returns a dict of arrays."""
    n, S = len(regions), len(sectors)

    names = names_params
    mats = {name: load_matrix(name, year, regions, sectors) for name in names}

    absorption = mats['dd0_'] + mats['nd0_'] + mats['m0_']
    absorption_safe = np.where(absorption < 1e-10, 1.0, absorption)

    id0_tensor = (params['id0_'][params['id0_']['yr'] == year]
                  .groupby(['r', 'g', 's'])['value'].sum()
                  .unstack('s')
                  .reindex(pd.MultiIndex.from_product([regions, sectors], names=['r', 'g']),
                           fill_value=0.0)
                  .reindex(columns=sectors, fill_value=0.0)
                  .fillna(0.0)
                  .values
                  .reshape(n, S, S))

    return {**mats, 'absorption': absorption, 'absorption_safe': absorption_safe,
            'id0': id0_tensor}

In [ ]:
EXCLUDED = {'fen', 'sle'}
sectors = sorted(s for s in params['xn0_']['g'].unique() if s not in EXCLUDED)

In [ ]:
# Index maps and dimensions
region_to_idx = {r: i for i, r in enumerate(regions)}
sector_to_idx = {s: i for i, s in enumerate(sectors)}
n, S = len(regions), len(sectors)
print(f'{n} regions x {S} sectors')

## Construction functions

In [ ]:
def ras_robust(seed, X, M, max_iter=2000, tol=1e-8):
    """RAS with convergence tracking. RAS lets us preserve the table's initial aggregate structure at the start.
    Indeed the approximation of the formula T = X.M.D^gamma distorts the matrix and does not guarantee that the resulting table
    is consistent with the aggregate flows observed at the start,
    i.e. that the total of what leaves as good g from state i toward states j equals the exports of state i for good g
    toward the NP in the initial table (row agreement);
    and that the total of what enters as good g into state j from states i equals the imports of state j for good g
    from the NP in the initial table (column agreement).

    The RAS algorithm proportionally scales up a whole row and a whole column at each iteration,
    until the row and column totals are close enough to the target totals (X and M).

    Args:
    seed: starting matrix (nxn)
    X: vector of row totals (n,)
    M: vector of column totals (n,)
    max_iter: maximum number of iterations
    tol: convergence tolerance

    Returns:
        T: adjusted matrix
        converged: boolean indicating whether convergence was reached for each sector
        final_err: final error (max of the deviations from the totals)
        iters: number of iterations performed
        initial_err: initial error (before adjustment)
    """
    T = seed.copy().astype(float) #starting nxn matrix created from the gravity seed with the chosen gamma
    r = X.values if hasattr(X, 'values') else X
    c = M.values if hasattr(M, 'values') else M
    initial_err = max(np.abs(T.sum(axis=1) - r).max(),
                      np.abs(T.sum(axis=0) - c).max())
    for it in range(max_iter):
        rs = T.sum(axis=1); rs[rs == 0] = 1
        T *= (r / rs)[:, None]
        cs = T.sum(axis=0); cs[cs == 0] = 1
        T *= (c / cs)[None, :]
        err = max(np.abs(T.sum(axis=1) - r).max(),
                  np.abs(T.sum(axis=0) - c).max())
        if err < tol:
            return T, True, err, it + 1, initial_err
    return T, False, err, max_iter, initial_err

In [ ]:
def reconstruct_bilateral_2(xn0_mat, nd0_mat, nm0_mat, sectors, sector_to_idx, regions,
                          D_np, gamma=1.0, imbalance_skip=0.50):
    """
    Reconstruct bilateral trade matrices T(region i→ region j, good g) via gravity model + RAS.
    Import target = nd0 + nm0 (direct absorption + margin absorption).
    Export target = xn0 (exports to national pool).
    Returns :
    T_all (dict sector→n×n array, n=regions) 
    df_log (convergence log).
    """
    n = len(regions)

    with np.errstate(divide='ignore', invalid='ignore'):
        friction = np.where(np.isnan(D_np), 0.0, D_np ** (-gamma))

    T_all = {}
    log_ras = []

    for g in sectors:
        g_i = sector_to_idx[g]
        X_g = pd.Series(xn0_mat[:, g_i] , index=regions)
        M_g = pd.Series(nd0_mat[:, g_i] + nm0_mat[:, g_i], index=regions) # for margin sectors, a part of their production is absorbed as margin (nm0) rather than direct use (nd0), but both contribute to the "import" side that RAS should match
        total_X = X_g.sum()
        total_M = M_g.sum()

        imbalance = abs(total_X - total_M) / total_X
        if total_M < 1e-10 or imbalance > imbalance_skip:
            T_all[g] = np.zeros((n, n))
            log_ras.append({'sector': g, 'status': 'skipped_imbalance',
                            'err': imbalance, 'iters': 0})
            continue

        #if imbalance > 1e-6:
            #M_g = M_g * (total_X / total_M)

        seed = np.outer(X_g.values, M_g.values) * friction
        seed += 1e-8 * np.outer(X_g.values / total_X, M_g.values / M_g.sum())
        np.fill_diagonal(seed, 0.0)

        T_g, converged, err, iters, initial_err = ras_robust(seed, X_g, M_g)
        
        tol_soft = 1e-6  # relaxed tolerance for borderline cases
        if converged:
            status = 'ok'
        elif err < tol_soft:
            status = 'ok_soft'   # converged to relaxed tolerance
        else:
            status = 'FAILED'

        T_all[g] = T_g
        log_ras.append({'sector': g, 'status': status, 'initial_err': initial_err,
                        'err': err, 'iters': iters, 'seed': 'gravity'})
        

    return T_all, pd.DataFrame(log_ras)


In [ ]:
def compute_use_shares_2(id0_df, cd0_mat, i0_mat, g0_mat, a0_mat=None):
    """
    Compute use shares (intermediate + final demand) at PURCHASER prices.

    For each (region r, good g):
        use_share_interm[r, g, s] = id0[r, g, s] / total_demand[r, g]
        use_share_C/I/G[r, g]     = cd0/i0/g0[r, g] / total_demand[r, g]
    with total_demand = id0.sum_s + cd0 + i0 + g0  (purchaser-price absorption).
    By construction the shares sum to 1 over {sectors s} + {C, I, G}.


    Fallback for margin goods where all demand is zero (id0=cd0=i0=g0=0):
    use_share_interm is set proportional to each sector's total intermediate
    purchases so that nm0/dm0 flows are not lost in build_Z.

    Arguments
    ---------
    id0_df  : (n, S, S) intermediary demand by (r,g,s)
    cd0_mat : (n, S)    household consumption demand by (r,g)
    i0_mat  : (n, S)    investment demand by (r,g)
    g0_mat  : (n, S)    government consumption demand by (r,g)
    a0_mat  : ignored (deprecated -- see note above)
    """
    # Purchaser-price absorption of good g in region r:
    # total intermediate demand (sum over buying sectors s) + final demand.
    total_demand = id0_df.sum(axis=2) + cd0_mat + i0_mat + g0_mat  # (n, S)
    safe = np.where(total_demand < 1e-10, 1.0, total_demand)       # (n, S)

    use_share_interm = id0_df  / safe[:, :, None]   # (n, S, S)
    use_share_C      = cd0_mat / safe               # (n, S)
    use_share_I      = i0_mat  / safe               # (n, S)
    use_share_G      = g0_mat  / safe               # (n, S)

    # --------------------------------------------------------------------------
    # FALLBACK FOR MARGIN GOODS WITH NO RECORDED DEMAND (id0=cd0=i0=g0=0).
    #
    # Margin goods (trade/transport) carry no demand of their own, so
    # total_demand(r,g)=0 and every use share above collapses to 0. The shares
    # would then sum to 0 and the margin flows (nm0/dm0), which build_Z spreads
    # through use_share_interm, would vanish from the table.
    #
    # Lacking any signal on who buys the margin good, we route it ENTIRELY to
    # intermediate use (0% to final demand) and split it across absorbing
    # sectors using the region's average intermediate-purchase profile.
    # --------------------------------------------------------------------------

    # Per sector s: its total intermediate demand across all goods g
    # (= column sum of the intermediate matrix). Shape (n, S).
    total_inputs = id0_df.sum(axis=1)

    # Region-wide total of intermediate purchases. Shape (n, 1).
    row_sum = total_inputs.sum(axis=1, keepdims=True)

    # "Representative buyer" profile: each sector's share of the region's total
    # intermediate purchases. Sums to 1 over s. Shape (n, S).
    fallback = total_inputs / np.where(row_sum < 1e-10, 1.0, row_sum)

    # Goods with zero total demand = the margin goods to patch. Shape (n, S, 1).
    mask_zero = (total_demand < 1e-10)[:, :, None]

    # For those goods only, overwrite the (over-s) interm shares with `fallback`
    # broadcast across the g axis. The interm shares now sum to 1 -> 100% of the
    # flow goes to intermediate demand; C/I/G stay 0 -> 0% to final demand.
    use_share_interm = np.where(mask_zero, fallback[:, np.newaxis, :], use_share_interm)

    return use_share_interm, use_share_C, use_share_I, use_share_G


In [ ]:
def build_Z(dd0_mat, nd0_mat, use_share_interm, T_all, sectors, n, S, xd0_mat=None):
    """
    xd0_mat : (n, S) supply to local market = dd0 + dm0.
              If None, falls back to dd0_mat (old behaviour, dm0 missing).
    Row sum  : xd0[r,g] + xn0[r,g] + x0[r,g]  = s0 (true WiNDC supply)
    """
    local_source = xd0_mat if xd0_mat is not None else dd0_mat

    Z_4d = np.zeros((n, S, n, S))

    for r_i in range(n):
        # xd0 = dd0 + dm0 : local margin revenue now included in row sums
        Z_4d[r_i, :, r_i, :] = local_source[r_i, :, None] * use_share_interm[r_i, :, :]

    for g_i, g in enumerate(sectors):
        T_g = T_all[g]
        ush = use_share_interm[:, g_i, :]
        Z_4d[:, g_i, :, :] += T_g[:, :, None] * ush[None, :, :]

    return Z_4d.reshape(n * S, n * S)


In [ ]:
def build_F(dd0_mat, T_all, use_share_C, use_share_I, use_share_G, sectors, n, S, xd0_mat=None):
    """
    Build final demand matrix F (n·S × n·3).
    Rows: (origin region, good). Columns: (destination region, {C, I, G}).
    """
    local_src = xd0_mat if xd0_mat is not None else dd0_mat  # ← only change
    
    F_4d = np.zeros((n, S, n, 3))

    for r_i in range(n):
        F_4d[r_i, :, r_i, 0] = local_src[r_i, :] * use_share_C[r_i, :]
        F_4d[r_i, :, r_i, 1] = local_src[r_i, :] * use_share_I[r_i, :]
        F_4d[r_i, :, r_i, 2] = local_src[r_i, :] * use_share_G[r_i, :]

    for g_i, g in enumerate(sectors):
        T_g = T_all[g]
        F_4d[:, g_i, :, 0] += T_g * use_share_C[:, g_i][None, :]  # fix: += not =
        F_4d[:, g_i, :, 1] += T_g * use_share_I[:, g_i][None, :]
        F_4d[:, g_i, :, 2] += T_g * use_share_G[:, g_i][None, :]

    return F_4d.reshape(n * S, n * 3)


def build_VA_EX(ld0_mat, kd0_mat, x0_mat, n, S):
    """Value added (labor + capital) and international exports, both flattened to n·S."""
    VA = (ld0_mat + kd0_mat).reshape(n * S)
    EX = x0_mat.reshape(n * S)
    return VA, EX


In [ ]:
def ras_2(Z0, u, v, max_iter=500, tol=1e-6):
    """
    Biproportional (RAS) scaling.
    Scales Z0 to match row sums u and column sums v.
    Requires sum(u) == sum(v); normalize v before calling if needed.
    """
    Z = Z0.copy().astype(float)
    for _ in range(max_iter):
        row_sums = Z.sum(axis=1)
        r_scale = np.divide(u, row_sums, out=np.zeros_like(u), where=row_sums > 1e-10)
        Z = Z * r_scale[:, None]

        col_sums = Z.sum(axis=0)
        s_scale = np.divide(v, col_sums, out=np.zeros_like(v), where=col_sums > 1e-10)
        Z = Z * s_scale[None, :]

        err = max(np.abs(Z.sum(axis=1) - u).max(),
                  np.abs(Z.sum(axis=0) - v).max())
        if err < tol:
            break
    return Z


In [ ]:
# Method switches for IOT construction. Toggle to compare construction variants.
DEFAULT_OPTS = {
    'basic_prices': True,                       # 1A : rescale id0 to basic prices in use shares
    'include_dm0':  False,                      # P0 : local supply = xd0 (dd0 + dm0) in Z & F instead of dd0
    'balance':      None,                       # 2A : column balancing onto ys0 -- None | 'colscale' | 'ras'
    'colscale_exclude': ('pip', 'trn', 'wht'),  # margin sectors whose supply rows colscale must not crush
}


def build_mriot_for_year(year, regions, sectors, sector_to_idx, D_np, gamma=1.0, opts=None):
    """
    Build one year of the WiNDC-derived MRIOT (v1.2 schema, margins included).

    Margin conventions (cf. build_IOT.ipynb cells 68-128):
      - md0[yr, r, m, g]: g is the DEMANDING good (good being margined)
      - nm0[yr, r, g, m] and dm0[yr, r, g, m]: g is the PROVIDING margin sector
      The market clearance Sigma_g md0(m) = Sigma_g (nm0+dm0)(m) is exact in WiNDC
      per (yr, r, m). Empirical analysis (cells 113, 121-122) shows xn0 ~= nm0 and
      dd0 ~= dm0 for margin sectors -- their production is essentially margin services.
      reconstruct_bilateral_2 (cell 128) uses nd0+nm0 as the import marginal, so
      margins are ALREADY integrated in Z via the gravity + RAS step. The vectors
      saved below allow downstream code to *separate* margin from direct flows for
      OECD-style comparison (OECD keeps margins in the Supply table, not in Z).
    """
    opts = {**DEFAULT_OPTS, **(opts or {})}
    n, S = len(regions), len(sectors)
    year = type(params['ys0_']['yr'].iloc[0])(year)

    names = ['dd0_', 'nd0_', 'xn0_', 'm0_', 'x0_', 'cd0_', 'i0_', 'g0_']
    data = load_year_data(year, regions, sectors, names)

    def mat(param):
        df = params[param]
        dim = 'g' if 'g' in df.columns else 's'
        return (df[df['yr'] == year]
                .groupby(['r', dim])['value'].sum().unstack(dim)
                .reindex(index=regions, columns=sectors, fill_value=0.0)
                .fillna(0.0).values)

    nm0_mat, ys0_mat = mat('nm0_'), mat('ys0_')
    dm0_mat          = mat('dm0_')
    ld0, kd0         = mat('ld0_'), mat('kd0_')

    ty0_mat = mat('ty0_')
    tm0_mat = mat('tm0_')
    ta0_mat = mat('ta0_')
    rx0_mat = mat('rx0_')

    # Margin demand and supply per type m  -- needed for per-type balance check
    def _agg_by_m(param_name, m_filter=None):
        df = params[param_name][params[param_name]['yr'] == year]
        sub = df if m_filter is None else df[df['m'] == m_filter]
        if sub.empty:
            return np.zeros((n, S))
        return (sub.groupby(['r', 'g'])['value'].sum().unstack('g')
                .reindex(index=regions, columns=sectors, fill_value=0.0)
                .fillna(0.0).values)

    md0_df = params['md0_'][params['md0_']['yr'] == year]
    margin_types  = sorted(md0_df['m'].unique()) if not md0_df.empty else []
    md0_total_mat = _agg_by_m('md0_')                                              # (n, S) -- absorbed good
    md0_trd_mat   = _agg_by_m('md0_', 'trd') if 'trd' in margin_types else np.zeros((n, S))
    md0_trn_mat   = _agg_by_m('md0_', 'trn') if 'trn' in margin_types else np.zeros((n, S))

    nm0_trd_mat = _agg_by_m('nm0_', 'trd') if 'trd' in margin_types else np.zeros((n, S))
    nm0_trn_mat = _agg_by_m('nm0_', 'trn') if 'trn' in margin_types else np.zeros((n, S))
    dm0_trd_mat = _agg_by_m('dm0_', 'trd') if 'trd' in margin_types else np.zeros((n, S))
    dm0_trn_mat = _agg_by_m('dm0_', 'trn') if 'trn' in margin_types else np.zeros((n, S))

    # Bilateral trade -- unchanged. Margin integration via nd0+nm0 import marginal
    # (solution 1 of build_IOT cell 119; validated by cell 121's Pearson/cosine analysis).
    T_all, df_log = reconstruct_bilateral_2(
        data['xn0_'], data['nd0_'], nm0_mat, sectors, sector_to_idx, regions, D_np, gamma=gamma)

    a0_mat = data['absorption'] if opts['basic_prices'] else None
    use_share_interm, use_share_C, use_share_I, use_share_G = compute_use_shares_2(
        data['id0'], data['cd0_'], data['i0_'], data['g0_'], a0_mat=a0_mat)

    xd0_mat = mat('xd0_') if opts['include_dm0'] else None

    Z = build_Z(data['dd0_'], data['nd0_'], use_share_interm, T_all, sectors, n, S, xd0_mat=xd0_mat)
    F = build_F(data['dd0_'], T_all, use_share_C, use_share_I, use_share_G, sectors, n, S, xd0_mat=xd0_mat)
    VA, EX = build_VA_EX(ld0, kd0, data['x0_'], n, S)

    M_total = data['m0_'].reshape(n * S)
    M_interm_mat = (data['m0_'][:, :, None] * use_share_interm).sum(axis=1)
    M_interm = M_interm_mat.reshape(n * S)
    M_fd_C   = (data['m0_'] * use_share_C).reshape(n * S)
    M_fd_I   = (data['m0_'] * use_share_I).reshape(n * S)
    M_fd_G   = (data['m0_'] * use_share_G).reshape(n * S)

    tax_prod_mat    = ty0_mat * ys0_mat
    tax_import_mat  = tm0_mat * data['m0_']
    absorption_base = data['dd0_'] + data['nd0_'] + data['m0_'] - rx0_mat
    tax_abs_mat     = ta0_mat * absorption_base
    tax_prod   = tax_prod_mat.reshape(n * S)
    tax_import = tax_import_mat.reshape(n * S)
    tax_abs    = tax_abs_mat.reshape(n * S)
    taxes_all  = tax_prod + tax_import + tax_abs

    Y = ys0_mat.reshape(n * S)

    if opts['balance']:
        v = (ys0_mat - ld0 - kd0 - tax_prod_mat - tax_import_mat - tax_abs_mat
             - M_interm_mat).clip(0).reshape(n * S)
        col = Z.sum(axis=0)
        if opts['balance'] == 'colscale':
            scale = np.divide(v, col, out=np.ones_like(v), where=col > 1e-10)
            Z_pre = Z.copy()
            Z = Z * scale[None, :]
            excl = [list(sectors).index(s) for s in opts['colscale_exclude'] if s in sectors]
            if excl:
                keep = np.zeros((n, S), dtype=bool); keep[:, excl] = True
                Z[keep.reshape(n * S), :] = Z_pre[keep.reshape(n * S), :]
        elif opts['balance'] == 'ras':
            u = Z.sum(axis=1)
            v = v * u.sum() / max(v.sum(), 1e-10)
            Z = ras_2(Z, u, v)

    return {
        'Z': Z, 'F': F, 'VA': VA, 'EX': EX, 'Y': Y,
        'M': M_total, 'M_interm': M_interm,
        'M_fd_C': M_fd_C, 'M_fd_I': M_fd_I, 'M_fd_G': M_fd_G,
        'tax_prod': tax_prod, 'tax_import': tax_import, 'tax_abs': tax_abs,
        'taxes': taxes_all,
        # Margins -- aggregate vectors (axis convention noted in docstring)
        'margins_demand':         md0_total_mat.reshape(n * S),         # per (r, absorbed good)
        'margins_demand_trd':     md0_trd_mat.reshape(n * S),
        'margins_demand_trn':     md0_trn_mat.reshape(n * S),
        'margins_supply_nat':     nm0_mat.reshape(n * S),               # per (r, providing sector)
        'margins_supply_dom':     dm0_mat.reshape(n * S),
        'margins_supply_nat_trd': nm0_trd_mat.reshape(n * S),
        'margins_supply_nat_trn': nm0_trn_mat.reshape(n * S),
        'margins_supply_dom_trd': dm0_trd_mat.reshape(n * S),
        'margins_supply_dom_trn': dm0_trn_mat.reshape(n * S),
        'margin_types':           np.array(margin_types),
        # Metadata for in-memory diagnose
        'regions': np.array(regions),
        'sectors': np.array(sectors),
        'log': df_log,
    }


In [ ]:
def diagnose_mriot(mriot, label='', top_margin=5):
    """Cost-identity check with full tax decomposition AND margin treatment.

    Identities reported:
      (A) Direct-output: Y_row vs Y_col using Y = ys0 (margin sectors will show
          a negative residual ~ -(nm0+dm0) because ys0 excludes margin revenue).
      (B) Full-output:   Y_full = Y + nm0 + dm0. Since reconstruct_bilateral_2
          integrates nm0 in the bilateral T (solution 1, cells 113-128), Z already
          carries the margin flows -- so Y_full should match Y_row / Y_col within
          RAS tolerance for margin sectors too.
      (C) Margin market clearance per type m: Sigma_g md0(m) vs Sigma_s (nm0+dm0)(m).
          Exact by WiNDC convention (cell 72), reported as a sanity check.

    Also reports `margin_intensity_top` = the top-`top_margin` sectors by
    (nm0+dm0)/ys0 ratio (cell 188-style breakdown).
    """
    Z, F, VA, EX = mriot['Z'], mriot['F'], mriot['VA'], mriot['EX']
    M_interm = mriot.get('M_interm', np.zeros_like(VA))

    # Tax components
    if 'tax_prod' in mriot:
        taxes_total   = mriot['tax_prod'] + mriot['tax_import'] + mriot['tax_abs']
        tax_breakdown = {
            'tax_prod':   round(float(mriot['tax_prod'].sum()),   1),
            'tax_import': round(float(mriot['tax_import'].sum()), 1),
            'tax_abs':    round(float(mriot['tax_abs'].sum()),    1),
        }
    else:
        taxes_total   = mriot.get('taxes', np.zeros_like(VA))
        tax_breakdown = {}

    Y_row = Z.sum(axis=1) + F.sum(axis=1) + EX
    Y_col = Z.sum(axis=0) + M_interm + VA + taxes_total

    # (A) Direct-output view
    mask     = Y_row > 0.1
    residual = Y_row[mask] - Y_col[mask]
    gap_pct  = np.abs(residual).sum() / Y_row[mask].sum() * 100
    n_neg    = int((residual < -1e-3).sum())
    n_nan    = int(np.isnan(Z).sum() + np.isnan(F).sum() + np.isnan(VA).sum())

    out = {
        'label': label, 'gap%': round(gap_pct, 2),
        'n_neg': n_neg, 'n_nan': n_nan,
        'Z_sum': round(float(Z.sum()), 1), 'VA_sum': round(float(VA.sum()), 1),
        'M_interm': round(float(M_interm.sum()), 1),
        'taxes_total': round(float(taxes_total.sum()), 1),
        **tax_breakdown,
    }

    Y_stored = mriot.get('Y', None)
    if Y_stored is not None:
        m2 = Y_stored > 0.1
        out['row_gap%'] = round(
            np.abs(Y_stored[m2] - Y_row[m2]).sum() / Y_stored[m2].sum() * 100, 2)
        out['col_gap%'] = round(
            np.abs(Y_stored[m2] - Y_col[m2]).sum() / Y_stored[m2].sum() * 100, 2)

    # (B) Full-output view with margins folded into output
    has_margins_supply = ('margins_supply_nat' in mriot and 'margins_supply_dom' in mriot)
    if Y_stored is not None and has_margins_supply:
        margins_supply_total = (mriot['margins_supply_nat']
                                + mriot['margins_supply_dom'])
        Y_full = Y_stored + margins_supply_total
        m3 = Y_full > 0.1
        out['margins_supply_sum'] = round(float(margins_supply_total.sum()), 1)
        out['row_gap_full%'] = round(
            np.abs(Y_full[m3] - Y_row[m3]).sum() / Y_full[m3].sum() * 100, 2)
        out['col_gap_full%'] = round(
            np.abs(Y_full[m3] - Y_col[m3]).sum() / Y_full[m3].sum() * 100, 2)

    # (C) Margin market clearance per type m (aggregate national level)
    if 'margins_demand' in mriot and has_margins_supply:
        md_sum = float(mriot['margins_demand'].sum())
        ms_sum = float((mriot['margins_supply_nat'] + mriot['margins_supply_dom']).sum())
        out['margins_demand_sum'] = round(md_sum, 1)
        out['margin_balance_gap%'] = round(
            abs(md_sum - ms_sum) / max(abs(md_sum), abs(ms_sum), 1) * 100, 2)

        for mtype in ('trd', 'trn'):
            kd = f'margins_demand_{mtype}'
            kn = f'margins_supply_nat_{mtype}'
            kdm = f'margins_supply_dom_{mtype}'
            if all(k in mriot for k in (kd, kn, kdm)):
                d_m = float(mriot[kd].sum())
                s_m = float(mriot[kn].sum() + mriot[kdm].sum())
                out[f'margin_gap_{mtype}%'] = round(
                    abs(d_m - s_m) / max(abs(d_m), abs(s_m), 1) * 100, 2)

    # M split sanity
    if all(k in mriot for k in ('M', 'M_fd_C', 'M_fd_I', 'M_fd_G')):
        M_check = mriot['M_interm'] + mriot['M_fd_C'] + mriot['M_fd_I'] + mriot['M_fd_G']
        out['M_split_err_max'] = round(float(np.abs(M_check - mriot['M']).max()), 4)

    # Margin intensity per sector: (nm0 + dm0) / ys0 -- reproduces cell 188 logic
    if (has_margins_supply and Y_stored is not None
            and 'sectors' in mriot and 'regions' in mriot):
        sec_names = [str(s) for s in np.asarray(mriot['sectors'])]
        n_sec = len(sec_names)
        n_r   = len(mriot['regions'])
        # Aggregate over states
        ms_per_sec = (mriot['margins_supply_nat'] + mriot['margins_supply_dom']
                      ).reshape(n_r, n_sec).sum(axis=0)
        Y_per_sec  = Y_stored.reshape(n_r, n_sec).sum(axis=0)
        with np.errstate(divide='ignore', invalid='ignore'):
            intensity = np.where(Y_per_sec > 0.1, ms_per_sec / Y_per_sec * 100, 0.0)
        idx_top = np.argsort(intensity)[::-1][:top_margin]
        out['margin_intensity_top'] = {
            sec_names[i]: round(float(intensity[i]), 1) for i in idx_top
        }

    status = 'OK'
    if n_nan:
        status = 'NaN'
    elif n_neg:
        status = f'NEG({n_neg})'
    elif gap_pct > 8 and out.get('col_gap_full%', 99) > 8:
        status = 'WARN'
    out['status'] = status
    return out


def run_pipeline(years, regions, sectors, sector_to_idx, D_np, gamma=1.0,
                 opts=None, output_dir=None, diagnose=True, verbose=True):
    """Build MRIOT tables for `years` (v1.2 schema with margins)."""
    import os, time
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    rows = []
    for year in years:
        t0 = time.time()
        mriot = build_mriot_for_year(year, regions, sectors, sector_to_idx, D_np,
                                     gamma=gamma, opts=opts)
        if output_dir:
            np.savez_compressed(
                os.path.join(output_dir, f'IOT_{year}.npz'),
                Z=mriot['Z'], F=mriot['F'], VA=mriot['VA'], EX=mriot['EX'],
                Y=mriot['Y'],
                M=mriot['M'], M_interm=mriot['M_interm'],
                M_fd_C=mriot['M_fd_C'], M_fd_I=mriot['M_fd_I'], M_fd_G=mriot['M_fd_G'],
                tax_prod=mriot['tax_prod'], tax_import=mriot['tax_import'],
                tax_abs=mriot['tax_abs'], taxes=mriot['taxes'],
                margins_demand=mriot['margins_demand'],
                margins_demand_trd=mriot['margins_demand_trd'],
                margins_demand_trn=mriot['margins_demand_trn'],
                margins_supply_nat=mriot['margins_supply_nat'],
                margins_supply_dom=mriot['margins_supply_dom'],
                margins_supply_nat_trd=mriot['margins_supply_nat_trd'],
                margins_supply_nat_trn=mriot['margins_supply_nat_trn'],
                margins_supply_dom_trd=mriot['margins_supply_dom_trd'],
                margins_supply_dom_trn=mriot['margins_supply_dom_trn'],
                margin_types=mriot['margin_types'],
                regions=np.array(regions), sectors=np.array(sectors))

        row = {'year': year}
        if diagnose:
            row.update(diagnose_mriot(mriot))
        log = mriot['log']
        row['ras_ok']     = int(log['status'].isin(['ok', 'ok_soft']).sum())
        row['ras_failed'] = int((log['status'] == 'FAILED').sum())
        row['t_s']        = round(time.time() - t0, 1)
        rows.append(row)
        if verbose:
            extra = ''
            if 'row_gap%' in row:
                extra += f"  row={row['row_gap%']}%  col={row['col_gap%']}%"
            if 'col_gap_full%' in row:
                extra += f"  col_full={row['col_gap_full%']}%"
            if 'margin_balance_gap%' in row:
                extra += f"  margin_bal={row['margin_balance_gap%']}%"
            print(f"{year}: gap={row.get('gap%', float('nan'))}%{extra}"
                  f"  ras_ok={row['ras_ok']:2d}  failed={row['ras_failed']}"
                  f"  ({row['t_s']}s)", flush=True)

    return pd.DataFrame(rows)


def compare_configs(years, regions, sectors, sector_to_idx, D_np, configs, gamma=1.0):
    """Build & diagnose several method configs on the same year(s) without saving."""
    years = [years] if isinstance(years, int) else list(years)
    rows = []
    for name, opts in configs.items():
        for year in years:
            mriot = build_mriot_for_year(year, regions, sectors, sector_to_idx, D_np,
                                         gamma=gamma, opts=opts)
            d = diagnose_mriot(mriot, label=name); d['year'] = year
            rows.append(d)
            print(f"  {name:>20} {year}: gap={d['gap%']:.2f}%  neg={d['n_neg']}", flush=True)

    df = pd.DataFrame(rows)
    print('\ngap% by config x year:')
    print(df.pivot(index='label', columns='year', values='gap%').to_string())
    return df


def diagnose_sector(mriot, regions, sectors, sector_name):
    """Per-sector row vs col balance check (used to inspect margin-heavy sectors).

    Aggregates across all states for the chosen sector and reports:
      - Y_row = Σ_r (Z.row + F.row + EX)[r, sector]
      - Y_col = Σ_r (Z.col + M_interm + VA + taxes)[r, sector]
      - gap%  = |Y_row - Y_col| / max(|Y_row|, 1) * 100
    """
    if sector_name not in sectors:
        return {'sector': sector_name, 'gap%': None, 'Y_row': None, 'Y_col': None}
    sec_list = list(sectors)
    s_idx = sec_list.index(sector_name)
    n, S  = len(regions), len(sec_list)

    Z, F, VA, EX = mriot['Z'], mriot['F'], mriot['VA'], mriot['EX']
    M_interm = mriot.get('M_interm', np.zeros_like(VA))
    if 'tax_prod' in mriot:
        taxes = mriot['tax_prod'] + mriot['tax_import'] + mriot['tax_abs']
    else:
        taxes = mriot.get('taxes', np.zeros_like(VA))

    Y_row = (Z.sum(axis=1) + F.sum(axis=1) + EX).reshape(n, S)
    Y_col = (Z.sum(axis=0) + M_interm + VA + taxes).reshape(n, S)

    yr = float(Y_row[:, s_idx].sum())
    yc = float(Y_col[:, s_idx].sum())
    gap = abs(yr - yc) / max(abs(yr), 1.0) * 100.0

    return {
        'sector': sector_name,
        'gap%':   round(gap, 2),
        'Y_row':  round(yr, 1),
        'Y_col':  round(yc, 1),
    }


## Pipeline run

In [ ]:
# Builds the MRIOT for the selected years. The SAVE switch decides whether the tables are
# written to disk; VERSION sets the dataset name under BASE_DIR (run_pipeline only writes
# when output_dir is not None).
GAMMA = 1.0
OPTS  = {'basic_prices': True, 'include_dm0': True, 'balance': 'ras'}

SAVE     = False                 # <-- toggle to test saving a version
VERSION  = 'grav_fric_v1.2'      # <-- adapt the name to the variant being saved
BASE_DIR = str(ROOT / 'data/interim/IOT/IOT_USA')
OUTPUT_DIR = f'{BASE_DIR}/{VERSION}' if SAVE else None

ALL_YEARS = sorted(params['xn0_']['yr'].unique())
YEARS     = ALL_YEARS[:1]        # quick test on 1 year; use ALL_YEARS[:] for the full run

summary = run_pipeline(YEARS, regions, sectors, sector_to_idx, D_np,
                       gamma=GAMMA, opts=OPTS, output_dir=OUTPUT_DIR)
print()
print(f'SAVE={SAVE}' + (f'  ->  written to {OUTPUT_DIR}' if SAVE else '  (in-memory only, nothing written)'))
print(summary.to_string(index=False))

## Quick diagnostic (single year)

In [ ]:
# In-memory build of one year + identity diagnostic, incl. margin-heavy sectors.
YEAR = 2017
m = build_mriot_for_year(YEAR, regions, sectors, sector_to_idx, D_np, gamma=GAMMA, opts=OPTS)
diag = diagnose_mriot(m, label=f'{VERSION} {YEAR}')
print({k: diag[k] for k in ('label', 'gap%', 'row_gap%', 'col_gap%', 'status') if k in diag})

for s in ('wht', 'trk', 'trn', 'pip'):
    print(diagnose_sector(m, regions, sectors, s))